# 02 — Analytical Base Table (ABT)
**Projeto:** Retail Forecast Intelligence — Previsão de Demanda com MLOps  
**Objetivo:** Documentar a tabela analítica final usada no treinamento — suas colunas, distribuições, completude e a lógica de cada feature.

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

ROOT = Path('..')
PROCESSED = ROOT / 'data' / 'processed'
ARTIFACTS = ROOT / 'artifacts'

print('Setup OK')

## 1. Carregamento da ABT

In [ ]:
abt = pd.read_parquet(PROCESSED / 'dataset.parquet')
abt['Date'] = pd.to_datetime(abt['Date'])

print(f'Shape: {abt.shape[0]:,} linhas × {abt.shape[1]} colunas')
print(f'Período: {abt["Date"].min().date()} → {abt["Date"].max().date()}')
print(f'Lojas:   {abt["Store"].nunique()}')
print(f'Depts:   {abt["Dept"].nunique()}')
abt.head(3)

## 2. Dicionário de dados — todas as colunas

In [ ]:
dicionario = {
    # Identificadores
    'Store':         ('Identificador da loja (1-45)', 'ID'),
    'Dept':          ('Identificador do departamento (1-99)', 'ID'),
    'Date':          ('Data da semana (sexta-feira)', 'Temporal'),
    # Target
    'Weekly_Sales':  ('Vendas semanais em USD — variável alvo', 'Target'),
    # Loja
    'Type':          ('Tipo da loja: A (grande), B (média), C (pequena)', 'Loja'),
    'Size':          ('Área da loja em m²', 'Loja'),
    # Contexto externo
    'IsHoliday':     ('1 se a semana contém feriado especial', 'Externo'),
    'Temperature':   ('Temperatura média regional (°F)', 'Externo'),
    'Fuel_Price':    ('Preço médio do combustível na região (USD/galão)', 'Externo'),
    'CPI':           ('Índice de preços ao consumidor da região', 'Externo'),
    'Unemployment':  ('Taxa de desemprego regional (%)', 'Externo'),
    'MarkDown1':     ('Desconto promocional tipo 1 (USD) — muitos nulos', 'Promoção'),
    'MarkDown2':     ('Desconto promocional tipo 2 (USD)', 'Promoção'),
    'MarkDown3':     ('Desconto promocional tipo 3 (USD)', 'Promoção'),
    'MarkDown4':     ('Desconto promocional tipo 4 (USD)', 'Promoção'),
    'MarkDown5':     ('Desconto promocional tipo 5 (USD)', 'Promoção'),
    # Features de calendário
    'year':          ('Ano extraído da data', 'Calendário'),
    'month':         ('Mês (1-12)', 'Calendário'),
    'weekofyear':    ('Semana do ano (1-52)', 'Calendário'),
    'dayofweek':     ('Dia da semana (0=segunda, 4=sexta)', 'Calendário'),
    'is_month_start':('1 se é início de mês', 'Calendário'),
    'is_month_end':  ('1 se é fim de mês', 'Calendário'),
    'week_sin':      ('Componente seno da semana — captura sazonalidade circular', 'Sazonal'),
    'week_cos':      ('Componente cosseno da semana — captura sazonalidade circular', 'Sazonal'),
    # Lags de vendas
    'lag_1':         ('Vendas da semana anterior (t-1)', 'Lag'),
    'lag_2':         ('Vendas de 2 semanas atrás (t-2)', 'Lag'),
    'lag_3':         ('Vendas de 3 semanas atrás (t-3)', 'Lag'),
    'lag_4':         ('Vendas de 4 semanas atrás (t-4)', 'Lag'),
    # Rolling stats
    'roll_mean_4':   ('Média móvel de 4 semanas (excl. semana atual)', 'Rolling'),
    'roll_std_4':    ('Desvio padrão móvel de 4 semanas', 'Rolling'),
    'roll_mean_8':   ('Média móvel de 8 semanas', 'Rolling'),
    'roll_std_8':    ('Desvio padrão móvel de 8 semanas', 'Rolling'),
    'roll_mean_12':  ('Média móvel de 12 semanas', 'Rolling'),
    'roll_std_12':   ('Desvio padrão móvel de 12 semanas', 'Rolling'),
    # Delta
    'diff_1':        ('Variação semanal: vendas_atual − vendas_semana_anterior', 'Delta'),
}

df_dict = pd.DataFrame([
    {'Coluna': col, 'Descrição': desc, 'Grupo': grupo,
     'Dtype': str(abt[col].dtype) if col in abt.columns else 'N/A',
     'Nulos (%)': f"{abt[col].isna().mean()*100:.1f}%" if col in abt.columns else 'N/A'}
    for col, (desc, grupo) in dicionario.items()
])

print(f'Total de features: {len(df_dict)}')
df_dict

## 3. Completude das features

In [ ]:
nulos = abt.isnull().mean().sort_values(ascending=False) * 100
nulos_significativos = nulos[nulos > 0]

if len(nulos_significativos) == 0:
    print('✅ Nenhuma coluna com nulos na ABT após o pipeline de preparação.')
else:
    print('Colunas com nulos:')
    print(nulos_significativos.to_string())
    fig, ax = plt.subplots(figsize=(10, 4))
    nulos_significativos.plot.bar(ax=ax, color='tomato')
    ax.set_title('Percentual de nulos por coluna')
    ax.set_ylabel('%')
    plt.tight_layout()
    plt.show()

## 4. Distribuição das features numéricas

In [ ]:
feature_cols = [c for c in abt.columns if c.startswith(('lag_', 'roll_', 'diff_', 'week_'))]

n_cols = 4
n_rows = (len(feature_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes_flat = axes.flatten()

for i, col in enumerate(feature_cols):
    data = abt[col].dropna()
    axes_flat[i].hist(data, bins=40, color='steelblue', edgecolor='white', linewidth=0.2)
    axes_flat[i].set_title(col, fontsize=10)
    axes_flat[i].set_yticks([])

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Distribuição das features de lag e rolling', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlação entre features e target

In [ ]:
num_cols = [c for c in abt.select_dtypes(include='number').columns
            if c not in ['Store', 'Dept', 'year', 'month', 'weekofyear', 'dayofweek',
                         'is_month_start', 'is_month_end', 'IsHoliday']]

corr_target = abt[num_cols].corrwith(abt['Weekly_Sales']).abs().sort_values(ascending=False)
corr_target = corr_target[corr_target.index != 'Weekly_Sales'].head(20)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#1d4ed8' if v > 0.7 else '#3b82f6' if v > 0.4 else '#93c5fd' for v in corr_target.values]
ax.barh(corr_target.index[::-1], corr_target.values[::-1], color=colors[::-1])
ax.axvline(0.4, color='orange', linestyle='--', linewidth=1, label='Limiar moderado (0.4)')
ax.axvline(0.7, color='red', linestyle='--', linewidth=1, label='Limiar forte (0.7)')
ax.set_title('Correlação absoluta das features com Weekly_Sales (target)')
ax.set_xlabel('|Pearson r|')
ax.legend()
plt.tight_layout()
plt.show()

> **Insight:** Os lags e rolling means têm correlação muito alta com o target (esperado — vendas são auto-correlacionadas). Isso confirma que são boas features preditivas. Variáveis externas (Temperature, Fuel_Price, CPI) têm correlação mais baixa mas adicionam contexto não capturado pelos lags.

## 6. Feature Importance dos modelos treinados

In [ ]:
fi_dir = ARTIFACTS / 'feature_importance'
modelos = [d.name for d in fi_dir.iterdir() if d.is_dir()] if fi_dir.exists() else []

fig, axes = plt.subplots(1, len(modelos), figsize=(6 * len(modelos), 7))
if len(modelos) == 1:
    axes = [axes]

for ax, modelo in zip(axes, modelos):
    fi_path = fi_dir / modelo / 'feature_importance.csv'
    if not fi_path.exists():
        continue
    fi = pd.read_csv(fi_path).head(15)
    fi_sorted = fi.sort_values('importance', ascending=True)
    ax.barh(fi_sorted['feature'], fi_sorted['importance'], color='steelblue')
    ax.set_title(f'Feature Importance\n{modelo.upper()}', fontsize=11)
    ax.set_xlabel('Importância')

plt.suptitle('Top 15 features por modelo', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7. Leaderboard dos modelos — resultados do Rolling CV

In [ ]:
lb_path = ARTIFACTS / 'reports' / 'leaderboard.csv'
lb = pd.read_csv(lb_path)

display_cols = [c for c in ['model', 'rmse', 'mae', 'smape', 'train_seconds'] if c in lb.columns]
lb_display = lb[display_cols].copy()
if 'smape' in lb_display.columns:
    lb_display['smape'] = lb_display['smape'].apply(lambda x: f'{x:.2f}%')
if 'train_seconds' in lb_display.columns:
    lb_display['train_seconds'] = lb_display['train_seconds'].apply(lambda x: f'{x:.0f}s')

print('Leaderboard — métricas médias do Rolling CV (3 folds):')
lb_display

In [ ]:
# Visualização comparativa
lb_num = pd.read_csv(lb_path)
metrics = [c for c in ['rmse', 'mae', 'smape'] if c in lb_num.columns]

fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 4))
if len(metrics) == 1:
    axes = [axes]

colors = ['#1d4ed8', '#3b82f6', '#93c5fd']
for ax, metric in zip(axes, metrics):
    bars = ax.bar(lb_num['model'], lb_num[metric], color=colors[:len(lb_num)])
    ax.set_title(metric.upper())
    ax.set_ylabel(metric.upper() + (' (%)' if metric == 'smape' else ' (USD)'))
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, lb_num[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:.0f}' if metric != 'smape' else f'{val:.2f}%',
                ha='center', fontsize=9)

plt.suptitle('Comparação de modelos — Rolling CV (média de 3 folds)', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Análise do conjunto de validação — qualidade das previsões

In [ ]:
vp_path = ARTIFACTS / 'reports' / 'valid_predictions.parquet'
vp = pd.read_parquet(vp_path)
vp['Date'] = pd.to_datetime(vp['Date'])
vp['erro'] = vp['Weekly_Sales'] - vp['y_pred']
vp['erro_abs'] = vp['erro'].abs()
denom = (vp['Weekly_Sales'].abs() + vp['y_pred'].abs()).replace(0, np.nan)
vp['smape_row'] = 2 * vp['erro_abs'] / denom * 100

print(f'Validação: {vp.shape[0]:,} registros | {vp["Date"].min().date()} → {vp["Date"].max().date()}')
print(f'SMAPE:  {vp["smape_row"].mean():.2f}%')
print(f'MAE:    ${vp["erro_abs"].mean():,.0f}')
print(f'RMSE:   ${np.sqrt((vp["erro"]**2).mean()):,.0f}')
print(f'Viés:   ${vp["erro"].mean():,.0f} (+ = subprevisão, - = superprevisão)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Real vs Previsto
sample = vp.sample(min(3000, len(vp)), random_state=42)
sc = axes[0].scatter(sample['Weekly_Sales'], sample['y_pred'],
                     c=sample['erro_abs'], cmap='RdYlGn_r',
                     alpha=0.3, s=8)
max_val = max(sample['Weekly_Sales'].max(), sample['y_pred'].max())
axes[0].plot([0, max_val], [0, max_val], 'k--', linewidth=1, alpha=0.5, label='Previsão perfeita')
axes[0].set_xlabel('Vendas reais (USD)')
axes[0].set_ylabel('Vendas previstas (USD)')
axes[0].set_title('Real vs Previsto')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
plt.colorbar(sc, ax=axes[0], label='Erro absoluto')

# Distribuição do erro
axes[1].hist(vp['erro'], bins=60, color='steelblue', edgecolor='white', linewidth=0.2)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Sem erro')
axes[1].axvline(vp['erro'].mean(), color='orange', linestyle='--', linewidth=1.5,
                label=f'Viés médio: ${vp["erro"].mean():,.0f}')
axes[1].set_xlabel('Erro (Real − Previsto) em USD')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição do Erro')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

plt.tight_layout()
plt.show()

## 9. Resumo da ABT

| Atributo | Valor |
|---|---|
| **Fonte** | Walmart Store Sales Forecasting (Kaggle) |
| **Granularidade** | Loja × Departamento × Semana |
| **Período** | Fev/2010 → Out/2012 (~143 semanas) |
| **Total de registros** | ~418k (após remoção de NaN em lag_1) |
| **Variável alvo** | `Weekly_Sales` — vendas semanais em USD |
| **Features de calendário** | 8 (year, month, week, dow, is_start, is_end, sin, cos) |
| **Features de lag** | 4 (lag_1 a lag_4) |
| **Features de rolling** | 6 (mean e std para janelas 4, 8, 12 semanas) |
| **Features externas** | 8 (temperatura, combustível, markdowns, CPI, desemprego, feriado) |
| **Features de loja** | 2 (tipo one-hot, tamanho) |
| **Total de features** | ~29 |
| **Estratégia de validação** | Rolling Time Series CV (3 folds, 28 dias por fold) |
| **Modelos treinados** | LightGBM, XGBoost, Random Forest |
| **Seleção do melhor modelo** | Média do RMSE nos 3 folds (não o melhor fold isolado) |